# Full-Dataset Autoencoder — Local Run

**Dataset:** Edge-IIoTset — `DNN-EdgeIIoT-dataset.csv` (all ~2.2M rows)

**Target machine:** 16 GB RAM, Intel i7 12th gen. No GPU required.

---

### What differs from the Colab version

| Change | Reason |
|---|---|
| Columns are selected *before* reading | Never loads the heavy text columns into RAM |
| `float32` instead of `float64` | Halves memory, no accuracy cost here |
| Baselines score a capped sample | One-Class SVM cost grows with the square of the row count |
| Reads a local path | No Drive mount, no upload |


## 0 — Configuration

Set `CSV_PATH` to wherever the extracted CSV lives on this machine.
Use a raw string (`r'...'`) so Windows backslashes are not treated as escapes.


In [ ]:
CSV_PATH = r'C:\Users\khaled\Desktop\PoC\DNN-EdgeIIoT-dataset.csv'

# None = every row. Set a number to work on a sample instead.
SAMPLE_ROWS = None

TEST_SPLIT    = 0.2    # share of normal traffic held out
THRESHOLD_PCT = 95     # error percentile above which a record is flagged

EPOCHS     = 20
BATCH_SIZE = 1024      # larger than Colab's 256 — faster on a strong CPU
SEED       = 42

# Baselines are scored on at most this many rows (One-Class SVM is O(n^2))
BASELINE_CAP = 50_000

OUT_DIR = 'PoC_results'

import os, gc, time
import numpy as np, random
np.random.seed(SEED); random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(CSV_PATH), f'CSV not found at: {CSV_PATH}'
print('CSV size:', round(os.path.getsize(CSV_PATH)/1e9, 2), 'GB')


## 1 — Decide which columns to read

The file has 63 columns, but roughly a third hold free text — IP addresses, URIs,
payloads. Those are useless to the model *and* expensive in memory, since pandas
stores each string as a Python object.

So instead of loading everything and dropping columns afterwards, we read a small
slice first, work out which columns are numeric, and then read only those.


In [ ]:
import pandas as pd

peek = pd.read_csv(CSV_PATH, nrows=2000, low_memory=False)
print('Total columns in file:', len(peek.columns))

LABEL_COL = 'Attack_label'
TYPE_COL  = 'Attack_type' if 'Attack_type' in peek.columns else None
assert LABEL_COL in peek.columns, 'Attack_label column missing'

# Identifier columns: numeric in places, but meaningless across networks
ID_LIKE = {'frame.time', 'ip.src_host', 'ip.dst_host',
           'arp.src.proto_ipv4', 'arp.dst.proto_ipv4',
           'http.file_data', 'http.request.full_uri', 'http.request.version',
           'icmp.transmit_timestamp', 'tcp.options', 'tcp.payload',
           'mqtt.msg', 'dns.qry.name', 'http.referer', 'mqtt.topic'}

numeric_cols = [col for col in peek.columns
                if col not in (LABEL_COL, TYPE_COL)
                and col not in ID_LIKE
                and pd.api.types.is_numeric_dtype(peek[col])]

keep = numeric_cols + [c for c in (LABEL_COL, TYPE_COL) if c]

print(f'Numeric features to load : {len(numeric_cols)}')
print(f'Columns skipped entirely : {len(peek.columns) - len(keep)}')
print()
print(numeric_cols)

del peek; gc.collect()


## 2 — Load the data

Read in chunks, so peak memory stays near one chunk rather than the whole file.

Two things matter here:

- **`usecols`** — the skipped text columns are never built in memory at all
- **per-chunk conversion to `float32`** — 4 bytes per number instead of 8

> ⚠ A column can look numeric in the first 2,000 rows and still hold text further
> down — `tcp.flags`, for instance, switches to hex strings like `0x00000018` once
> the attack rows begin. Declaring `dtype=float32` up front crashes on those.
> Each chunk is therefore converted with `errors='coerce'`: anything unparseable
> becomes `NaN` and is counted, so nothing fails silently.

Expect a minute or two for the full file.


In [ ]:
t0 = time.time()
CHUNK = 200_000

frac = None
if SAMPLE_ROWS is not None:
    # The rows are ordered, normal traffic first, so reading the first N would
    # return a single class. Sample evenly from every chunk instead.
    total = sum(1 for _ in open(CSV_PATH, encoding='utf-8', errors='ignore')) - 1
    frac = min(1.0, SAMPLE_ROWS / total)
    print(f'Rows in file: {total:,} — sampling {frac:.1%}')

parts = []
unparsable = {col: 0 for col in numeric_cols}

for chunk in pd.read_csv(CSV_PATH, usecols=keep, chunksize=CHUNK, low_memory=False):
    for col in numeric_cols:
        conv = pd.to_numeric(chunk[col], errors='coerce')
        unparsable[col] += int(conv.isna().sum() - chunk[col].isna().sum())
        chunk[col] = conv.astype(np.float32)
    chunk[LABEL_COL] = (pd.to_numeric(chunk[LABEL_COL], errors='coerce')
                          .fillna(0).astype(np.int8))
    if frac is not None:
        chunk = chunk.sample(frac=frac, random_state=SEED)
    parts.append(chunk)

df = pd.concat(parts, ignore_index=True)
del parts; gc.collect()

print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} columns in {time.time()-t0:.0f}s')
print(f'DataFrame memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')


### Columns that turned out not to be numeric

Blank cells were already `NaN` before conversion and are not counted here, so any
count below means the column holds **actual text** — a hex code, a flag name, a
URL fragment. Such a column is text, not a measurement, and is dropped.

The cut-off is 1% of rows: generous enough to tolerate a handful of malformed
lines, strict enough to catch a column that is text in disguise.


In [ ]:
n = len(df)
TEXT_TOLERANCE = 0.01          # >1% unparseable means the column is text

bad = {col: v for col, v in unparsable.items() if v}

if bad:
    print('Columns containing non-numeric text:')
    for col, v in sorted(bad.items(), key=lambda kv: -kv[1]):
        flag = 'DROP' if v > TEXT_TOLERANCE * n else 'keep'
        print(f'  [{flag}] {col:<30} {v:>10,} of {n:,}  ({v/n:.2%})')
else:
    print('Every selected column parsed cleanly as numeric.')

dead = [col for col, v in unparsable.items() if v > TEXT_TOLERANCE * n]
if dead:
    print(f'\nDropping {len(dead)} text columns: {dead}')
    numeric_cols = [col for col in numeric_cols if col not in dead]

print(f'\nFeatures carried forward: {len(numeric_cols)}')
assert numeric_cols, 'No numeric features left — check the CSV.'


In [ ]:
print(df[LABEL_COL].value_counts().rename({0: 'Normal', 1: 'Attack'}))
if TYPE_COL:
    print(); print(df[TYPE_COL].value_counts())

counts = df[LABEL_COL].value_counts()
assert 0 in counts.index and 1 in counts.index, (
    'Only one class present — check SAMPLE_ROWS.')
print('\nBoth classes present.')


## 3 — Final cleaning

The heavy work was done at read time. What remains is to neutralise infinities and
missing values, and to drop columns that never vary.


In [ ]:
X = df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

const_cols = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=const_cols)

print(f'Dropped {len(const_cols)} constant columns')
print(f'Final feature count: {X.shape[1]}')
if const_cols:
    print('Dropped:', const_cols)


## 4 — Split and scale

> ⚠ The scaler is fitted on training data only. Test data is transformed, never
> fitted — otherwise information about the attacks leaks into preprocessing.

> ⚠ The scaled attack values are **not** clipped to 0–1. They fall outside that
> range precisely because they are anomalous, and that is the signal the model
> detects. Clipping them collapses detection.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

y = df[LABEL_COL].values
attack_types = df.loc[y == 1, TYPE_COL].values if TYPE_COL else None

X_tr, X_te_norm = train_test_split(X[y == 0], test_size=TEST_SPLIT, random_state=SEED)
X_attack = X[y == 1]

scaler = MinMaxScaler()
Xtr_s  = scaler.fit_transform(X_tr).astype(np.float32)
Xten_s = scaler.transform(X_te_norm).astype(np.float32)
Xatk_s = scaler.transform(X_attack).astype(np.float32)

n_feat = Xtr_s.shape[1]

# Free the pandas objects — only the arrays are needed from here
del X, X_tr, X_te_norm, X_attack
df_small = df[[LABEL_COL] + ([TYPE_COL] if TYPE_COL else [])].copy()
del df; gc.collect()

print(f'Train (normal) : {len(Xtr_s):,}')
print(f'Test  normal   : {len(Xten_s):,}')
print(f'Test  attack   : {len(Xatk_s):,}')
print(f'Features       : {n_feat}  |  dtype: {Xtr_s.dtype}')
print(f'Attack range   : {Xatk_s.min():.2f} -> {Xatk_s.max():.2f}')


## 5 — Build the autoencoder

Architecture `n → n/2 → n/4 → n/2 → n`. The bottleneck forces the network to learn
patterns rather than memorise rows.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

print('TensorFlow', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU devices:', gpus if gpus else 'none — running on CPU (fine for this model)')

tf.random.set_seed(SEED)

h1, h2 = max(32, n_feat // 2), max(16, n_feat // 4)

inp = Input(shape=(n_feat,))
e   = Dense(h1, activation='relu')(inp)
e   = Dense(h2, activation='relu')(e)
d   = Dense(h1, activation='relu')(e)
out = Dense(n_feat, activation='sigmoid')(d)

ae = Model(inp, out)
ae.compile(optimizer=Adam(1e-3), loss='mse')
print(f'\nArchitecture: {n_feat} -> {h1} -> {h2} -> {h1} -> {n_feat}')
ae.summary()


## 6 — Train

Input and target are the same array: the model's only job is to reproduce what it
is given. On the full dataset expect roughly 15–40 minutes on CPU.

`EarlyStopping` halts training once validation loss stops improving, so a longer
`EPOCHS` costs nothing when the model converges early.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

t0 = time.time()
hist = ae.fit(
    Xtr_s, Xtr_s,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=True,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True,
                             monitor='val_loss')],
    verbose=1
)
print(f'\nTraining took {(time.time()-t0)/60:.1f} minutes')


In [ ]:
import matplotlib
matplotlib.use('Agg')          # write files without opening windows
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist.history['loss'],     label='Train',      lw=2)
ax.plot(hist.history['val_loss'], label='Validation', lw=2, ls='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss')
ax.set_title('Autoencoder training')
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig1_training.png', dpi=150)
print('saved fig1_training.png')


## 7 — Reconstruction error and threshold

Errors are computed in batches so no oversized intermediate array is ever built.


In [ ]:
def recon_error(model, X, batch=20_000):
    """Mean squared reconstruction error per row, computed batch by batch."""
    out = np.empty(len(X), dtype=np.float32)
    for i in range(0, len(X), batch):
        chunk = X[i:i+batch]
        R = model.predict(chunk, batch_size=4096, verbose=0)
        out[i:i+batch] = np.mean(np.square(chunk - R), axis=1)
    return out

t0 = time.time()
err_train = recon_error(ae, Xtr_s)
err_norm  = recon_error(ae, Xten_s)
err_atk   = recon_error(ae, Xatk_s)
print(f'Scored {len(err_train)+len(err_norm)+len(err_atk):,} rows'
      f' in {time.time()-t0:.0f}s')

threshold = float(np.percentile(err_train, THRESHOLD_PCT))
print(f'\nThreshold (p{THRESHOLD_PCT}) = {threshold:.6f}')
print(f'Mean error, normal : {err_norm.mean():.6f}')
print(f'Mean error, attack : {err_atk.mean():.6f}')
print(f'Ratio              : {err_atk.mean()/max(err_norm.mean(),1e-12):.1f}x')


## 8 — Results


In [ ]:
tp = int((err_atk  > threshold).sum())
fn = int((err_atk <= threshold).sum())
fp = int((err_norm > threshold).sum())
tn = int((err_norm<= threshold).sum())

detection = tp / (tp + fn)
false_pos = fp / (fp + tn)
precision = tp / max(tp + fp, 1)
f1        = 2*precision*detection / max(precision + detection, 1e-12)
accuracy  = (tp + tn) / (tp + tn + fp + fn)

print('=' * 44)
print(f"{'Attack samples':<24}{tp+fn:>12,}")
print(f"{'Detected':<24}{tp:>12,}")
print(f"{'Missed':<24}{fn:>12,}")
print('-' * 44)
print(f"{'Detection rate':<24}{detection:>11.2%}")
print(f"{'False positive rate':<24}{false_pos:>11.2%}")
print(f"{'Precision':<24}{precision:>11.2%}")
print(f"{'F1 score':<24}{f1:>11.2%}")
print(f"{'Accuracy':<24}{accuracy:>11.2%}")
print('=' * 44)


### Key figure — normal vs attack error distribution


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

lo = float(min(err_norm.min(), err_atk.min()))
hi = float(np.percentile(np.concatenate([err_norm[:200_000],
                                         err_atk[:200_000]]), 99.5))
bins = np.linspace(lo, hi, 70)

ax.hist(err_norm, bins=bins, alpha=.65, label='Normal', color='#2a9d8f', density=True)
ax.hist(err_atk,  bins=bins, alpha=.65, label='Attack', color='#e76f51', density=True)
ax.axvline(threshold, color='#264653', ls='--', lw=2,
           label=f'Threshold (p{THRESHOLD_PCT})')
ax.set_xlabel('Reconstruction error'); ax.set_ylabel('Density')
ax.set_title(f'Normal vs attack — detection rate {detection:.1%}')
ax.legend(); ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig2_separation.png', dpi=150)
print('saved fig2_separation.png')


## 9 — Detection rate per attack type

The most informative table in the whole run: it shows which attack families the
method handles and which it does not.


In [ ]:
if TYPE_COL:
    rows = []
    for t in pd.unique(attack_types):
        m = attack_types == t
        rows.append({'Attack type': t,
                     'Samples' : int(m.sum()),
                     'Detected': int((err_atk[m] > threshold).sum()),
                     'Rate'    : float((err_atk[m] > threshold).mean())})
    per_type = pd.DataFrame(rows).sort_values('Rate', ascending=False)
    per_type['Rate'] = (per_type['Rate'] * 100).round(2)
    display(per_type.reset_index(drop=True))

    d = per_type.sort_values('Rate')
    fig, ax = plt.subplots(figsize=(7, .38*len(d) + 1.6))
    ax.barh(d['Attack type'], d['Rate'], color='#e76f51')
    ax.set_xlabel('Detection rate (%)'); ax.set_xlim(0, 100)
    ax.set_title('Detection rate per attack type')
    ax.grid(axis='x', alpha=.25)
    fig.tight_layout()
    fig.savefig(f'{OUT_DIR}/fig3_per_attack.png', dpi=150)
    print('saved fig3_per_attack.png')


## 10 — Baseline comparison

> ⚠ **One-Class SVM scales with the square of the row count.** Scoring hundreds of
> thousands of rows with it can run for hours. Both baselines are therefore fitted
> and scored on a capped random sample — the autoencoder keeps its full-data score,
> and the sample is large enough for a fair comparison.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

rng = np.random.default_rng(SEED)

def cap(A, n):
    if len(A) <= n:
        return A
    return A[rng.choice(len(A), n, replace=False)]

fit_sample  = cap(Xtr_s,  BASELINE_CAP)
norm_sample = cap(Xten_s, BASELINE_CAP)
atk_sample  = cap(Xatk_s, BASELINE_CAP)
print(f'Baselines use {len(fit_sample):,} train / {len(atk_sample):,} attack rows')

t0 = time.time()
iso = IsolationForest(n_estimators=100, contamination=0.05,
                      random_state=SEED, n_jobs=-1).fit(fit_sample)
print(f'Isolation Forest fitted in {time.time()-t0:.0f}s')

t0 = time.time()
svm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale').fit(fit_sample[:10_000])
print(f'One-Class SVM fitted in {time.time()-t0:.0f}s')


In [ ]:
def rates(model):
    """scikit-learn marks anomalies as -1."""
    return ((model.predict(atk_sample)  == -1).mean(),
            (model.predict(norm_sample) == -1).mean())

iso_d, iso_f = rates(iso)
svm_d, svm_f = rates(svm)

comp = pd.DataFrame([
    {'Method': 'Autoencoder',      'Detection %': detection*100,
     'False positive %': false_pos*100, 'Scored rows': len(Xatk_s)},
    {'Method': 'Isolation Forest', 'Detection %': iso_d*100,
     'False positive %': iso_f*100,     'Scored rows': len(atk_sample)},
    {'Method': 'One-Class SVM',    'Detection %': svm_d*100,
     'False positive %': svm_f*100,     'Scored rows': len(atk_sample)},
]).round(2)

display(comp)


> **If a baseline scores higher, that is expected — and here is the answer:**
>
> *Isolation Forest performs well on blatant attacks such as DDoS and scanning,
> where a single record is already statistically extreme. But it evaluates every
> record independently — it has no notion of time.*
>
> *The attacks this project targets — sensor reading manipulation and replay —
> look perfectly normal in any single record. What exposes them is the sequence
> over time, which requires an LSTM and is absent from Isolation Forest.*


## 11 — Save everything


In [ ]:
import json

summary = {
    'dataset'            : os.path.basename(CSV_PATH),
    'rows_used'          : int(len(Xtr_s) + len(Xten_s) + len(Xatk_s)),
    'features_used'      : int(n_feat),
    'train_normal'       : int(len(Xtr_s)),
    'test_normal'        : int(len(Xten_s)),
    'test_attack'        : int(len(Xatk_s)),
    'epochs_run'         : int(len(hist.history['loss'])),
    'threshold'          : threshold,
    'detection_rate'     : float(detection),
    'false_positive_rate': float(false_pos),
    'precision'          : float(precision),
    'f1'                 : float(f1),
    'accuracy'           : float(accuracy),
}

with open(f'{OUT_DIR}/summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)

comp.to_csv(f'{OUT_DIR}/comparison.csv', index=False)
if TYPE_COL:
    per_type.to_csv(f'{OUT_DIR}/per_attack_type.csv', index=False)

ae.save(f'{OUT_DIR}/autoencoder.keras')

import pickle
with open(f'{OUT_DIR}/scaler.pkl', 'wb') as fh:
    pickle.dump(scaler, fh)

print('Saved to', os.path.abspath(OUT_DIR))
print(json.dumps(summary, indent=2))


---

## Done

| File | Contents |
|---|---|
| `fig1_training.png` | Training and validation loss |
| `fig2_separation.png` | **Key figure** — normal vs attack errors |
| `fig3_per_attack.png` | Detection rate by attack type |
| `comparison.csv` | Autoencoder vs the two baselines |
| `per_attack_type.csv` | The per-family breakdown |
| `summary.json` | Every metric |
| `autoencoder.keras` | The trained model |
| `scaler.pkl` | The fitted scaler — needed to score any new data |

`scaler.pkl` matters: scoring new traffic later requires the *same* scaler that was
fitted here. Re-fitting a fresh one would produce different, incomparable numbers.

---

### Reference

Ferrag, M. A., Friha, O., Hamouda, D., Maglaras, L., Janicke, H. (2022).
*Edge-IIoTset: A New Comprehensive Realistic Cyber Security Dataset of IoT and
IIoT Applications for Centralized and Federated Learning.* IEEE Access.
DOI: 10.1109/ACCESS.2022.3165809
